In [1]:
import pandas as pd
import numpy as np
import random

In [2]:
lpi = pd.read_csv('/content/International_LPI_Scorecard_Select_Countries_v2.csv')
combined_ppi = pd.read_csv('/content/combined_ppi_data.csv')
commodity_prices = pd.read_csv('/content/monthly_world_bank_commodity_prices_1960_2025_v2.csv')
tariff = pd.read_csv('/content/tarriff_database_2025_only_semiconductor_components_v2.csv')

In [ ]:
viable_countries = lpi.country_code.unique()

In [ ]:
additional_viable = combined_ppi.country_code.unique()

In [ ]:
tariff.head()

,hts8,brief_description,mfn_text_rate_pct,how_measured
0,84145915,"Fans used for cooling microprocessors, telecom...",0.0,NO
1,84248910,"Mechanical appliances for projecting, dispersi...",0.0,NO
2,84733011,"Printed circuit assemblies, not incorporating ...",0.0,NaN
3,84733020,Parts and accessories of the ADP machines of h...,0.0,NO
4,84734010,Printed circuit assemblies for automatic telle...,0.0,NO


# Extracting Unique Country Codes


In [3]:
unique_lpi_countries = set(lpi['country_code'].unique())
unique_combined_ppi_countries = set(combined_ppi['country_code'].unique())
# The 'tariff' DataFrame does not have a 'country_code' column, so it is excluded from this step.

all_unique_countries = unique_lpi_countries.union(
    unique_combined_ppi_countries
)

unique_countries = sorted(list(all_unique_countries))
print(f"Total unique countries identified: {len(unique_countries)}")
print(f"Unique countries: {unique_countries}")

Total unique countries identified: 21
Unique countries: ['ARE', 'AUS', 'BEL', 'BRA', 'CAN', 'CHN', 'DEU', 'FIN', 'FRA', 'GBR', 'HKG', 'IDN', 'IND', 'JPN', 'MEX', 'MYS', 'NLD', 'OAC', 'SGP', 'THA', 'USA']


# FRED USA

Will be used to model foreign suppliers in the EU because this reflects the prices of foreign-produced components imported into the U.S.

This ensure domestic and foreign suppliers follow the correct inflation dynamics

# Mapping

## Integrated Circuit Parts

Only keeping data points up to 2023 so all countries have values (BLS IC only up to 2023)



In [ ]:
circuits = pd.read_csv('/content/bls_ppi_integrated_circuit_packages_1976_2023_v2.csv')

In [ ]:
circuits

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year
0,1976-01-01,1976,1,357.300,PCU3344133344131,Semiconductor and related device manufacturing,Integrated circuit packages,BLS,USA,1998
1,1976-02-01,1976,2,355.300,PCU3344133344131,Semiconductor and related device manufacturing,Integrated circuit packages,BLS,USA,1998
2,1976-03-01,1976,3,349.900,PCU3344133344131,Semiconductor and related device manufacturing,Integrated circuit packages,BLS,USA,1998
3,1976-04-01,1976,4,351.500,PCU3344133344131,Semiconductor and related device manufacturing,Integrated circuit packages,BLS,USA,1998
4,1976-05-01,1976,5,351.900,PCU3344133344131,Semiconductor and related device manufacturing,Integrated circuit packages,BLS,USA,1998
...,...,...,...,...,...,...,...,...,...,...
571,2023-08-01,2023,8,21.858,PCU3344133344131,Semiconductor and related device manufacturing,Integrated circuit packages,BLS,USA,1998
572,2023-09-01,2023,9,21.818,PCU3344133344131,Semiconductor and related device manufacturing,Integrated circuit packages,BLS,USA,1998
573,2023-10-01,2023,10,21.635,PCU3344133344131,Semiconductor and related device manufacturing,Integrated circuit packages,BLS,USA,1998
574,2023-11-01,2023,11,21.757,PCU3344133344131,Semiconductor and related device manufacturing,Integrated circuit packages,BLS,USA,1998


In [ ]:
combined_ppi = pd.concat([combined_ppi.loc[lambda x: ~(x.source=='BLS') & (x.year <= 2023)], circuits]).reset_index(drop=True)

In [ ]:
# country-specific 1998 baselines for integrated circuit packages per unit
baseline_circuits = {
    'USA': 0.25, # Anchor; mature, high-cost manufacturing
    'JPN': 0.3, # ~1.2x US; high labor, high quality, stock packaging ecosystem
    'DEU': 0.28, # Germany; Slightly below Japan, advanced high-cost EU manufacturing
    'FRA': 0.27, # Similar to DEU; Western Europe cost structure
    'GBR': 0.27, # Similar Western Europe cost level
    'NLD': 0.27, # Western Europe, high-cost but efficient
    'BEL': 0.27, # Western Europe, similar to FRA/NLD
    'CAN': 0.24, # Slightly below US; similar structure, marginally lower costs
    'AUS': 0.24, # Developed, smaller electronics base, similar to CAN
    'FIN': 0.26, # Nordic, high labor cost, niche electronics
    'SGP': 0.22, # Advanced hub, efficient, slightly cheaper than US
    'HKG': 0.21, # Trade/assembly hub, lower labor than US/EU
    'MYS': 0.15, # strong electronics assembly
    'CHN': 0.13, # Very low labor + scale advantages in late 1990s
    'MEX': 0.17, # Nearshore, lower labor than US, higher than China/MYS
    'BRA': 0.18, # Higher friction than Asia, but lower labor than US/EU
    'IND': 0.16, # Low labor, less mature packaging ecosystem in 1998
    'IDN': 0.15, # Low labor, emerging electronics base
    'ARE': 0.26, # UAE; High-income, imported tech, niche local assembly
    'OAC': 0.18 # "Other Asian Countries/Offshore aggregate"; mid-low cost blend
}

## Mapping Countries to PPI tables

In [26]:
ppi_map = {
    'USA': ('BLS', 'USA'),
    'CAN': ('FRED', 'CAN'),
    'CHN': ('FRED', 'CHN'),
    'MEX': ('FRED', 'MEX'),
    'OAC': ('FRED', 'OAC'),

    # Countries mapped to OAC (Asian import PPI)
    'ARE': ('FRED', 'OAC'), # Gulf imports dominated by Asian electronics
    'AUS': ('FRED', 'OAC'), # Asia-Pacific supply chain
    'HKG': ('FRED', 'OAC'), # Part of Asian supply chain
    'IDN': ('FRED', 'OAC'), # Southeast Asia
    'IND': ('FRED', 'OAC'), # South Asia, similar to OAC basket
    'JPN': ('FRED', 'OAC'), # Part of Asian electronics ecosystem
    'MYS': ('FRED', 'OAC'),
    'SGP': ('FRED', 'OAC'),
    'THA': ('FRED', 'OAC'),

    # Countries mapped to FRED USA (global import PPI)
    # U.S. import PPI is closest match to European semiconductor cost dymanics
    'BEL': ('FRED', 'USA'), # Western Europe -> Similar inflation to US imprt
    'DEU': ('FRED', 'USA'), # EU inflation similar to US import PPIs
    'FIN': ('FRED', 'USA'), # EU electronics inflation similar to US
    'FRA': ('FRED', 'USA'),
    'GBR': ('FRED', 'USA'),
    'NLD': ('FRED', 'USA'),

    # Brazil still maps to OAC
    'BRA': ('FRED', 'OAC') # Brazil imports Asian semiconductors
}


## Adding PPI source and region columns based on country mapping

Tell model which PPI series to reference for scaling

In [ ]:
combined_ppi['ppi_source'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][0]
)
combined_ppi['ppi_region'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][1]
)

## Computing earliest data for each (ppi_source, ppi_region)

In [ ]:
# finds first availale month for each PPI series
start_dates = (
    combined_ppi
    .groupby(['source', 'country_code'])['date']
    .min()
)


In [ ]:
start_dates

source  country_code
BLS     USA             1976-01-01
FRED    CAN             2012-06-01
        CHN             2012-06-01
        MEX             2012-06-01
        OAC             2012-06-01
        USA             2005-12-01
Name: date, dtype: object

## Extracting PPI(start_date) for each region

In [ ]:
ppi_start = (
    combined_ppi
    .merge(start_dates.rename('start_date'),
           on=['source', 'country_code'])
    .query('date == start_date')
    .set_index(['source', 'country_code'])['ppi_value']
)


In [ ]:
ppi_start

source  country_code
FRED    CAN             100.0
        CHN             100.0
        MEX             100.0
        OAC             100.0
        USA             100.0
BLS     USA             357.3
Name: ppi_value, dtype: float64

# Overriding USA's anchor to use 1998

In [ ]:
ppi_usa_1998 = (
    combined_ppi
    .query("source == 'BLS' and country_code == 'USA' and year == 1998")
    ['ppi_value']
    .iloc[0]
)

ppi_start_corrected = ppi_start.copy()
ppi_start_corrected[('BLS', 'USA')] = ppi_usa_1998


## Building a suppler x date grid

where every date x every supplier_country_code (BEL, DEU, FIN, etc)

In [ ]:
supplier_countries = list(baseline_circuits.keys())
all_dates = combined_ppi['date'].drop_duplicates().sort_values()

supplier_grid = (
    all_dates.to_frame(name='date')
    .assign(key=1)
    .merge(
        pd.DataFrame({'country_code': supplier_countries, 'key': 1}),
        on='key'
    )
    .drop(columns='key')
)


In [ ]:
supplier_grid

,date,country_code
0,1976-01-01,USA
1,1976-01-01,JPN
2,1976-01-01,DEU
3,1976-01-01,FRA
4,1976-01-01,GBR
...,...,...
11515,2023-12-01,BRA
11516,2023-12-01,IND
11517,2023-12-01,IDN
11518,2023-12-01,ARE


# Mapping suppliers to PPI series using ppi_map

In [ ]:
supplier_grid['ppi_source'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][0])
supplier_grid['ppi_region'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][1])


# Attaching regional PPI values to the supplier grid

In [ ]:
ppi_series = combined_ppi[[
    'date',
    'source',
    'country_code',
    'ppi_value'
]].rename(columns={
    'source': 'ppi_source',
    'country_code': 'ppi_region'
})

supplier_with_ppi = supplier_grid.merge(
    ppi_series,
    on=['date', 'ppi_source', 'ppi_region'],
    how='left'
)



In [ ]:
# merging
supplier_with_ppi = supplier_grid.merge(
    ppi_series,
    on=['date', 'ppi_source', 'ppi_region'],
    how='left'
)


In [ ]:
supplier_with_ppi

,date,country_code,ppi_source,ppi_region,ppi_value
0,1976-01-01,USA,BLS,USA,357.3
1,1976-01-01,JPN,FRED,OAC,NaN
2,1976-01-01,DEU,FRED,USA,NaN
3,1976-01-01,FRA,FRED,USA,NaN
4,1976-01-01,GBR,FRED,USA,NaN
...,...,...,...,...,...
11515,2023-12-01,BRA,FRED,OAC,89.8
11516,2023-12-01,IND,FRED,OAC,89.8
11517,2023-12-01,IDN,FRED,OAC,89.8
11518,2023-12-01,ARE,FRED,OAC,89.8


## Computing real_price for each supplier x date

In [14]:
def compute_real_price_row(row, baseline_dict):
    country = row['country_code']          # supplier country
    source = row['ppi_source']
    region = row['ppi_region']

    baseline = baseline_dict[country]
    ppi_t = row['ppi_value']
    ppi_start_val = ppi_start_corrected[(source, region)]

    return baseline * (ppi_t / ppi_start_val)


In [ ]:
supplier_with_ppi['real_price_of_integrated_circuit_components'] = (
    supplier_with_ppi.apply(compute_real_price_row, axis=1)
)

In [ ]:
supplier_with_ppi.loc[lambda x: (x.ppi_source=='BLS') & (x.ppi_region=='USA') & (x.ppi_value == 100)]

,date,country_code,ppi_source,ppi_region,ppi_value,real_price_of_integrated_circuit_components
5480,1998-11-01,USA,BLS,USA,100.0,0.233863
5500,1998-12-01,USA,BLS,USA,100.0,0.233863


In [ ]:
ic_components = supplier_with_ppi

In [ ]:
ic_components.to_csv('ic_components_2023.csv', index=False)

# Microprocessors/Microcontrollers

# NOTE: Only have BLS for this data up to 2015

Base year 2007

(Relative to 1998)
Much higher-value add

Far more concentrated manufacturing
In 2007, the global MPU/MCU landscape was dominated by
 - USA (Intel, AMD, TI, Freescale)
 - Japan (Renesas, Toshiba, Fujitsu)
 - Taiwan (TSMC, UMC)
 - Europe (STMicro, Infineon, NXP)

 Packaging is labor-heavy. Microprocessors are fab-heavy.

 For 2007, a realistic USA baseline for a mid-range microcontroller die is ~ $1.20 per die.

 Scaling other countries relative to this.

In [ ]:
# country-specific 2007 baselines for mid-range microcontroller
baseline_microprocessors_2007 = {
    'USA': 1.20, # Anchor; Intel, AMD, TI, Freescale; high-cost domestic fabs
    'JPN': 1.35, # Renasas, Toshiba, Fujitsu; very high domestic fab costs
    'DEU': 1.30, # Infineon; high-cost EU fabs
    'FRA': 1.28, # STMicro; EU cost structure
    'GBR': 1.28, # ARM ecosystem; high engineering cost
    'NLD': 1.28, # NXP; EU fab/test cost structure
    'BEL': 1.28, # IMEC ecosystem; high R&D cost
    'CAN': 1.15, # Slightly lower overhead than US
    'AUS': 1.15, # Developed, smaller electronics base, similar to CAN
    'FIN': 1.26, # Nordic, high labor cost, niche electronics
    'SGP': 1.05, # Advanced fabs (GlobalFoundries), efficient operations
    'HKG': 1.00, # Regional hub; fabless ecosystem
    'MYS': 0.80, # Similar to MYS; strong electronics assembly
    'CHN': 0.75, # SMIC emergingl low labor; maturing fabs
    'MEX': 0.85, # Assembly/test; higher friction than Asia
    'BRA': 0.90, # Higher friction than Asia, but lower labor than US/EU; import heavy ecosystem
    'IND': 0.82, # Fabless ecosystem; low labor; limited fabs
    'IDN': 0.80, # Assembly/test; low labor
    'ARE': 1.25, # High-income, imported tech, niche fabs
    'OAC': 0.90 # Weighted blend of 'lesser' Asian countries
}

In [ ]:
microprocessors = pd.read_csv('/content/bls_ppi_microprocessors_and_microcontrollers_1981_2015_v2.csv')

In [ ]:
microprocessors

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year
0,1981-06-01,1981,6,308726.1,PCU33441333441312,Semiconductor and related device manufacturing,Microprocessors including microcontrollers,BLS,USA,2007
1,1981-07-01,1981,7,306760.4,PCU33441333441312,Semiconductor and related device manufacturing,Microprocessors including microcontrollers,BLS,USA,2007
2,1981-08-01,1981,8,306760.4,PCU33441333441312,Semiconductor and related device manufacturing,Microprocessors including microcontrollers,BLS,USA,2007
3,1981-09-01,1981,9,312037.8,PCU33441333441312,Semiconductor and related device manufacturing,Microprocessors including microcontrollers,BLS,USA,2007
4,1981-10-01,1981,10,312396.3,PCU33441333441312,Semiconductor and related device manufacturing,Microprocessors including microcontrollers,BLS,USA,2007
...,...,...,...,...,...,...,...,...,...,...
401,2014-11-01,2014,11,46.3,PCU33441333441312,Semiconductor and related device manufacturing,Microprocessors including microcontrollers,BLS,USA,2007
402,2014-12-01,2014,12,46.3,PCU33441333441312,Semiconductor and related device manufacturing,Microprocessors including microcontrollers,BLS,USA,2007
403,2015-01-01,2015,1,46.3,PCU33441333441312,Semiconductor and related device manufacturing,Microprocessors including microcontrollers,BLS,USA,2007
404,2015-02-01,2015,2,46.3,PCU33441333441312,Semiconductor and related device manufacturing,Microprocessors including microcontrollers,BLS,USA,2007


In [ ]:
combined_ppi = pd.concat([combined_ppi.loc[lambda x: ~(x.source=='BLS') & (x.year <=2015)],
           microprocessors]).reset_index(drop=True)

In [ ]:
combined_ppi['ppi_source'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][0]
)
combined_ppi['ppi_region'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][1]
)

# finds first availale month for each PPI series
start_dates = (
    combined_ppi
    .groupby(['source', 'country_code'])['date']
    .min()
)

ppi_start = (
    combined_ppi
    .merge(start_dates.rename('start_date'),
           on=['source', 'country_code'])
    .query('date == start_date')
    .set_index(['source', 'country_code'])['ppi_value']
)

ppi_usa_2007 = (
    combined_ppi
    .query("source == 'BLS' and country_code == 'USA' and year == 2007")
    ['ppi_value']
    .iloc[0]
)

ppi_start_corrected = ppi_start.copy()
ppi_start_corrected[('BLS', 'USA')] = ppi_usa_2007

supplier_countries = list(baseline_circuits.keys())
all_dates = combined_ppi['date'].drop_duplicates().sort_values()

supplier_grid = (
    all_dates.to_frame(name='date')
    .assign(key=1)
    .merge(
        pd.DataFrame({'country_code': supplier_countries, 'key': 1}),
        on='key'
    )
    .drop(columns='key')
)

# Add ppi_source and ppi_region to supplier_grid before merging
supplier_grid['ppi_source'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][0])
supplier_grid['ppi_region'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][1])

ppi_series = combined_ppi[[
    'date',
    'source',
    'country_code',
    'ppi_value'
]].rename(columns={
    'source': 'ppi_source',
    'country_code': 'ppi_region'
})

supplier_with_ppi = supplier_grid.merge(
    ppi_series,
    on=['date', 'ppi_source', 'ppi_region'],
    how='left'
)



In [ ]:
supplier_with_ppi

,date,country_code,ppi_source,ppi_region,ppi_value
0,1981-06-01,USA,BLS,USA,308726.1
1,1981-06-01,JPN,FRED,OAC,NaN
2,1981-06-01,DEU,FRED,USA,NaN
3,1981-06-01,FRA,FRED,USA,NaN
4,1981-06-01,GBR,FRED,USA,NaN
...,...,...,...,...,...
8295,2015-12-01,BRA,FRED,OAC,97.3
8296,2015-12-01,IND,FRED,OAC,97.3
8297,2015-12-01,IDN,FRED,OAC,97.3
8298,2015-12-01,ARE,FRED,OAC,97.3


In [ ]:
supplier_with_ppi['real_price_of_microprocessors'] = (
    supplier_with_ppi.apply(compute_real_price_row, axis=1)
)

In [ ]:
microprocessors = supplier_with_ppi

In [ ]:
microprocessors.to_csv('microprocessors_2015.csv', index=False)

# Transistors/Diodes

Up to 2025

In [20]:
transistors = pd.read_csv('/content/bls_ppi_transistors_diodes_lesserComponents_1977_2025_v2-2.csv')

In [5]:
transistors.tail()

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year
583,2025-08-01,2025,8,54.808,PCU334413334413A,Semiconductor and related device manufacturing,"Other semiconductor devices, including transis...",BLS,USA,1981
584,2025-09-01,2025,9,55.058,PCU334413334413A,Semiconductor and related device manufacturing,"Other semiconductor devices, including transis...",BLS,USA,1981
585,2025-10-01,2025,10,55.058,PCU334413334413A,Semiconductor and related device manufacturing,"Other semiconductor devices, including transis...",BLS,USA,1981
586,2025-11-01,2025,11,55.095,PCU334413334413A,Semiconductor and related device manufacturing,"Other semiconductor devices, including transis...",BLS,USA,1981
587,2025-12-01,2025,12,55.108,PCU334413334413A,Semiconductor and related device manufacturing,"Other semiconductor devices, including transis...",BLS,USA,1981


In [21]:
baseline_transistors_1981 = {
    'USA': 0.05,
    'JPN': 0.045,
    'DEU': 0.055,
    'FRA': 0.055,
    'GBR': 0.055,
    'NLD': 0.055,
    'BEL': 0.055,
    'FIN': 0.053,
    'CAN': 0.048,
    'AUS': 0.048,
    'SGP': 0.040,
    'HKG': 0.038,
    'MYS': 0.035,
    'THA': 0.035,
    'CHN': 0.030,
    'MEX': 0.040,
    'BRA': 0.042,
    'IND': 0.032,
    'IDN': 0.033,
    'ARE': 0.060,
    'OAC': 0.037
}


In [22]:
combined_ppi = pd.read_csv('/content/combined_ppi_data.csv')

In [23]:
combined_ppi = pd.concat([combined_ppi.loc[lambda x: ~(x.source=='BLS') & (x.year <=2025)],
           transistors]).reset_index(drop=True)

In [27]:
combined_ppi['ppi_source'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][0]
)
combined_ppi['ppi_region'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][1]
)

# finds first availale month for each PPI series
start_dates = (
    combined_ppi
    .groupby(['source', 'country_code'])['date']
    .min()
)

ppi_start = (
    combined_ppi
    .merge(start_dates.rename('start_date'),
           on=['source', 'country_code'])
    .query('date == start_date')
    .set_index(['source', 'country_code'])['ppi_value']
)

ppi_usa_1981 = (
    combined_ppi
    .query("source == 'BLS' and country_code == 'USA' and year == 1981")
    ['ppi_value']
    .iloc[0]
)

ppi_start_corrected = ppi_start.copy()
ppi_start_corrected[('BLS', 'USA')] = ppi_usa_1981

supplier_countries = list(baseline_transistors_1981.keys())
all_dates = combined_ppi['date'].drop_duplicates().sort_values()

supplier_grid = (
    all_dates.to_frame(name='date')
    .assign(key=1)
    .merge(
        pd.DataFrame({'country_code': supplier_countries, 'key': 1}),
        on='key'
    )
    .drop(columns='key')
)

# Add ppi_source and ppi_region to supplier_grid before merging
supplier_grid['ppi_source'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][0])
supplier_grid['ppi_region'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][1])

ppi_series = combined_ppi[[
    'date',
    'source',
    'country_code',
    'ppi_value'
]].rename(columns={
    'source': 'ppi_source',
    'country_code': 'ppi_region'
})

supplier_with_ppi = supplier_grid.merge(
    ppi_series,
    on=['date', 'ppi_source', 'ppi_region'],
    how='left'
)



In [28]:
supplier_with_ppi['real_price_of_transistors'] = (
    supplier_with_ppi.apply(compute_real_price_row, axis=1, args=(baseline_transistors_1981,))
)

In [29]:
supplier_with_ppi.tail(30)

,date,country_code,ppi_source,ppi_region,ppi_value,real_price_of_transistors
12318,2025-11-01,MYS,FRED,OAC,98.100,0.034335
12319,2025-11-01,THA,FRED,OAC,98.100,0.034335
12320,2025-11-01,CHN,FRED,CHN,NaN,NaN
12321,2025-11-01,MEX,FRED,MEX,78.800,0.031520
12322,2025-11-01,BRA,FRED,OAC,98.100,0.041202
12323,2025-11-01,IND,FRED,OAC,98.100,0.031392
12324,2025-11-01,IDN,FRED,OAC,98.100,0.032373
12325,2025-11-01,ARE,FRED,OAC,98.100,0.058860
12326,2025-11-01,OAC,FRED,OAC,98.100,0.036297
12327,2025-12-01,USA,BLS,USA,55.108,0.027720


In [31]:
supplier_with_ppi.to_csv('transistors_2025.csv', index=False)

# Power Devices

For this component, Malaysia, Thailand, and the Philippines are documented as global discrete hubs.

China/Taiwan are dominant sources.

Mexico only assembly and test.

Europe: automotive-grader, higher cost.

Japan = higher labor and higher quality.

In [ ]:
oac = pd.read_csv('/content/fred_ppi_other_asian_countries_semiconductor_manufacturing_2012_2025_v2.csv')

In [ ]:
oac

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year
0,2012-06-01,2012,6,100.0,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012
1,2012-07-01,2012,7,99.2,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012
2,2012-08-01,2012,8,99.1,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012
3,2012-09-01,2012,9,100.0,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012
4,2012-10-01,2012,10,100.1,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012
...,...,...,...,...,...,...,...,...,...,...
158,2025-08-01,2025,8,95.1,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012
159,2025-09-01,2025,9,95.3,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012
160,2025-10-01,2025,10,96.7,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012
161,2025-11-01,2025,11,98.1,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012


In [ ]:
oac_2012_base_value = oac[oac['year'] == 2012]['ppi_value'].iloc[0]
oac['growth_rate'] = (oac['ppi_value'] / oac_2012_base_value) - 1
display(oac.head())

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year,growth_rate
0,2012-06-01,2012,6,100.0,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,0.000
1,2012-07-01,2012,7,99.2,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,-0.008
2,2012-08-01,2012,8,99.1,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,-0.009
3,2012-09-01,2012,9,100.0,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,0.000
4,2012-10-01,2012,10,100.1,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,0.001


In [ ]:
china = pd.read_csv('/content/fred_ppi_china_semiconductor_manufacturing_2012_2018_v2.csv')

In [ ]:
china

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year
0,2012-06-01,2012,6,100.0,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012
1,2012-07-01,2012,7,101.0,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012
2,2012-08-01,2012,8,101.5,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012
3,2012-09-01,2012,9,101.4,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012
4,2012-10-01,2012,10,101.2,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012
...,...,...,...,...,...,...,...,...,...,...
74,2018-08-01,2018,8,89.0,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012
75,2018-09-01,2018,9,89.0,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012
76,2018-10-01,2018,10,89.1,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012
77,2018-11-01,2018,11,89.4,COCHNZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,CHN,2012


In [ ]:
last_china_row = china.iloc[-1]
last_china_ppi_value = last_china_row['ppi_value']
last_china_date = pd.to_datetime(last_china_row['date'])

# Static columns from the last row of china to be used for new rows
series_id = last_china_row['series_id']
industry = last_china_row['industry']
product = last_china_row['product']
source = last_china_row['source']
country_code = last_china_row['country_code']
base_year = last_china_row['base_year']

print(f"Last China PPI Value: {last_china_ppi_value}")
print(f"Last China Date: {last_china_date}")

Last China PPI Value: 89.4
Last China Date: 2018-12-01 00:00:00


In [ ]:
oac['growth_rate'] = oac['ppi_value'].pct_change()
oac['growth_rate'] = oac['growth_rate'].fillna(0)
display(oac.head())

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year,growth_rate
0,2012-06-01,2012,6,100.0,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,0.000000
1,2012-07-01,2012,7,99.2,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,-0.008000
2,2012-08-01,2012,8,99.1,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,-0.001008
3,2012-09-01,2012,9,100.0,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,0.009082
4,2012-10-01,2012,10,100.1,COOASZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,OAC,2012,0.001000


In [ ]:
oac_extension = oac[oac['date'] > last_china_date].copy()

current_ppi_value = last_china_ppi_value
extended_rows = []

for index, row in oac_extension.iterrows():
    new_ppi_value = current_ppi_value * (1 + row['growth_rate'])
    extended_rows.append({
        'date': row['date'].strftime('%Y-%m-%d'),
        'year': row['year'],
        'month': row['month'],
        'ppi_value': new_ppi_value,
        'series_id': series_id,
        'industry': industry,
        'product': product,
        'source': source,
        'country_code': country_code,
        'base_year': base_year
    })
    current_ppi_value = new_ppi_value

# Convert the list of dictionaries to a DataFrame
extended_china_df = pd.DataFrame(extended_rows, columns=china.columns)

# Concatenate the original china DataFrame and the new extended rows DataFrame
extended_china = pd.concat([china, extended_china_df], ignore_index=True)

print("Extended China DataFrame head:")
print(extended_china.head())
print("\nExtended China DataFrame tail:")
print(extended_china.tail())

Extended China DataFrame head:
                  date  year  month  ppi_value   series_id  \
0  2012-06-01 00:00:00  2012      6      100.0  COCHNZ3344   
1  2012-07-01 00:00:00  2012      7      101.0  COCHNZ3344   
2  2012-08-01 00:00:00  2012      8      101.5  COCHNZ3344   
3  2012-09-01 00:00:00  2012      9      101.4  COCHNZ3344   
4  2012-10-01 00:00:00  2012     10      101.2  COCHNZ3344   

                                            industry  \
0  Semiconductor and other electronic component m...   
1  Semiconductor and other electronic component m...   
2  Semiconductor and other electronic component m...   
3  Semiconductor and other electronic component m...   
4  Semiconductor and other electronic component m...   

                                         product source country_code  \
0  Semiconductor and other electronic components   FRED          CHN   
1  Semiconductor and other electronic components   FRED          CHN   
2  Semiconductor and other electronic compo

In [ ]:
mexico = pd.read_csv('/content/fred_ppi_mexico_computer_electronic_manufacturing_2012_2025_v2.csv')

In [ ]:
eu = pd.read_csv('/content/fred_ppi_united_states_semiconductor_manufacturing_2005_2025_v2.csv')

In [ ]:
major_components = pd.read_csv('/content/bls_ppi_aggregated_major_semiconductor_products_1981_2025_v2.csv')

In [ ]:
combined_ppi = pd.concat(
    [eu, mexico, extended_china, oac, major_components]
)

In [ ]:
combined_ppi

,date,year,month,ppi_value,series_id,industry,product,source,country_code,base_year,growth_rate
0,2005-12-01,2005,12,100.000,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,NaN
1,2006-01-01,2006,1,99.600,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,NaN
2,2006-02-01,2006,2,99.400,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,NaN
3,2006-03-01,2006,3,99.500,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,NaN
4,2006-04-01,2006,4,98.800,IZ3344,Semiconductor and other electronic component m...,Semiconductor and other electronic components,FRED,USA,2005,NaN
...,...,...,...,...,...,...,...,...,...,...,...
530,2025-08-01,2025,8,28.392,PCU334413334413P,Semiconductor and related device manufacturing,Primary products,BLS,USA,1998,NaN
531,2025-09-01,2025,9,28.454,PCU334413334413P,Semiconductor and related device manufacturing,Primary products,BLS,USA,1998,NaN
532,2025-10-01,2025,10,28.632,PCU334413334413P,Semiconductor and related device manufacturing,Primary products,BLS,USA,1998,NaN
533,2025-11-01,2025,11,28.639,PCU334413334413P,Semiconductor and related device manufacturing,Primary products,BLS,USA,1998,NaN


In [ ]:
baseline_power_devices_2012 = {
    'USA': 0.12,
    'JPN': 0.14,
    'DEU': 0.15,
    'FRA': 0.14,
    'GBR': 0.14,
    'NLD': 0.14,
    'SGP': 0.09,
    'MYS': 0.08,
    'THA': 0.078,
    'CHN': 0.07,
    'MEX': 0.095
}


In [ ]:
combined_ppi['ppi_source'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][0]
)
combined_ppi['ppi_region'] = combined_ppi['country_code'].map(
    lambda c: ppi_map[c][1]
)

# Convert 'date' column to datetime objects to ensure proper sorting
combined_ppi['date'] = pd.to_datetime(combined_ppi['date'])

# finds first availale month for each PPI series
start_dates = (
    combined_ppi
    .groupby(['source', 'country_code'])['date']
    .min()
)

ppi_start = (
    combined_ppi
    .merge(start_dates.rename('start_date'),
           on=['source', 'country_code'])
    .query('date == start_date')
    .set_index(['source', 'country_code'])['ppi_value']
)

ppi_usa_2012 = (
    combined_ppi
    .query("source == 'BLS' and country_code == 'USA' and year == 2012")
    ['ppi_value']
    .iloc[0]
)

ppi_start_corrected = ppi_start.copy()
ppi_start_corrected[('BLS', 'USA')] = ppi_usa_2012

supplier_countries = list(baseline_power_devices_2012.keys()) # Use baseline_power_devices_2012
all_dates = combined_ppi['date'].drop_duplicates().sort_values()

supplier_grid = (
    all_dates.to_frame(name='date')
    .assign(key=1)
    .merge(
        pd.DataFrame({'country_code': supplier_countries, 'key': 1}),
        on='key'
    )
    .drop(columns='key')
)

# Add ppi_source and ppi_region to supplier_grid before merging
supplier_grid['ppi_source'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][0])
supplier_grid['ppi_region'] = supplier_grid['country_code'].map(lambda c: ppi_map[c][1])

ppi_series = combined_ppi[[
    'date',
    'source',
    'country_code',
    'ppi_value'
]].rename(columns={
    'source': 'ppi_source',
    'country_code': 'ppi_region'
})

supplier_with_ppi = supplier_grid.merge(
    ppi_series,
    on=['date', 'ppi_source', 'ppi_region'],
    how='left'
)


In [ ]:
def compute_real_price_of_power_devices(row):
    return compute_real_price_row(row, baseline_power_devices_2012)


In [ ]:
supplier_with_ppi['real_price_of_power_devices'] = (
    supplier_with_ppi.apply(compute_real_price_of_power_devices, axis=1)
)


In [ ]:
supplier_with_ppi.drop(supplier_with_ppi.loc[supplier_with_ppi['country_code'] == 'USA'].index,axis=0, inplace=True)

In [ ]:
supplier_with_ppi.drop(supplier_with_ppi.loc[lambda x: pd.to_datetime(x.date) <= '2012'].index, axis=0, inplace= True)

In [ ]:
power_devices = supplier_with_ppi
power_devices.to_csv('power_devices_2025.csv', index=False)